In [42]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf


In [43]:
processed_train = pd.read_csv('../data/processed/processed_train.csv')
processed_test = pd.read_csv('../data/processed/processed_test.csv')

x_train = processed_train['clean_comment'].astype(str).to_numpy()
x_test = processed_test['clean_comment'].astype(str).to_numpy() 

y_train = processed_train['category'].map({-1:0,0:1,1:2}).astype("int32").to_numpy()
y_test = processed_test['category'].map({-1:0,0:1,1:2}).astype("int32").to_numpy()

In [44]:
from tensorflow.keras.layers import TextVectorization



vectorizer = TextVectorization(
    max_tokens=10000,
    output_mode='int',
    output_sequence_length=250
)

vectorizer.adapt(x_train)



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, Input, Bidirectional

from tensorflow.keras.layers import LSTM, GRU

model = Sequential([
    Input(shape=(1,),dtype='string'),
    vectorizer,
    Embedding(10000, 64),
    Bidirectional(GRU(28)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(28,activation='tanh'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])



In [45]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 250)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 250, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 56)             │        15,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         3,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 28)             │         1,820 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 28)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            87 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 661,347 (2.52 MB)

 Trainable params: 661,347 (2.52 MB)

 Non-trainable params: 0 (0.00 B)

In [46]:
model.fit(x_train, y_train, epochs=5, batch_size=32, validation_data=(x_test, y_test))

Epoch 1/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 139s 132ms/step - accuracy: 0.7569 - loss: 0.5935 - val_accuracy: 0.8735 - val_loss: 0.3469
Epoch 2/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 105s 113ms/step - accuracy: 0.9069 - loss: 0.2763 - val_accuracy: 0.8970 - val_loss: 0.3226
Epoch 3/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 100s 107ms/step - accuracy: 0.9315 - loss: 0.2125 - val_accuracy: 0.8913 - val_loss: 0.3427
Epoch 4/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 98s 105ms/step - accuracy: 0.9462 - loss: 0.1669 - val_accuracy: 0.8789 - val_loss: 0.3729
Epoch 5/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 107s 115ms/step - accuracy: 0.9590 - loss: 0.1307 - val_accuracy: 0.8771 - val_loss: 0.4295
